# EcoGrid-Twin: Dataset Preprocessing & Provenance Verification

This notebook audits all CSV datasets in `dataset/`, reporting:
- Record counts, timestamps, and sampling frequencies
- Data provenance (Real NASA observations vs Physics-simulated vs Synthetic)
- Missing value checks and integrity audit
- Time-series visualization of irradiance, temperature, solar PV, and building load.

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working Directory:', os.getcwd())

## 1. Audit All Dataset CSVs (Row Counts, Dates, Missing Values)

In [ ]:
csv_files = sorted(glob.glob('dataset/*.csv'))
audit_rows = []

for p in csv_files:
    name = os.path.basename(p)
    try:
        df = pd.read_csv(p, comment='#')
        n_rows = len(df)
        n_cols = len(df.columns)
        nulls = df.isnull().sum().sum()
        if 'timestamp' in df.columns:
            t_start = str(df['timestamp'].iloc[0])
            t_end = str(df['timestamp'].iloc[-1])
        else:
            t_start = 'N/A'
            t_end = 'N/A'
        audit_rows.append({
            'File': name,
            'Rows': n_rows,
            'Cols': n_cols,
            'Missing Values': nulls,
            'Start Date': t_start,
            'End Date': t_end
        })
    except Exception as e:
        audit_rows.append({'File': name, 'Error': str(e)})

audit_df = pd.DataFrame(audit_rows)
audit_df

## 2. Inspect Merged Microgrid Time Series (`merged_dataset.csv`)
Used as the primary ground-truth input for training the LSTM forecaster.

In [ ]:
df_merged = pd.read_csv('dataset/merged_dataset.csv')
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])
df_merged.set_index('timestamp', inplace=True)
df_merged.describe()

## 3. Visualize Sample 7-Day Microgrid Telemetry (July 2026)

In [ ]:
sample = df_merged.loc['2026-07-01':'2026-07-07']

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, dpi=120)

axes[0].plot(sample.index, sample['solar_output_kw'], color='#10b981', label='Solar PV (kW)')
axes[0].set_ylabel('Solar (kW)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right')

axes[1].plot(sample.index, sample['load_kw'], color='#a855f7', label='Building Load (kW)')
axes[1].set_ylabel('Load (kW)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right')

axes[2].plot(sample.index, sample['temperature'], color='#f59e0b', label='Temperature (°C)')
axes[2].set_ylabel('Temp (°C)')
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend(loc='upper right')

plt.suptitle('EcoGrid-Twin 7-Day Telemetry Sample (Chennai Microgrid)', fontsize=14)
plt.tight_layout()
plt.show()